# Упражнение: Разбор лога JSON-RPC: запросы, ответы, ошибки и таймауты

## Задание

MCP-подобный обмен между клиентом и сервером записан в `input.csv`: `line_no`, `ts_ms` (время сообщения
в миллисекундах), `direction` (`C->S` — от клиента серверу, `S->C` — от сервера клиенту) и `raw` —
само сообщение в формате JSON-RPC 2.0. Типы сообщений:

- **запрос** — есть `method` и `id`;
- **уведомление** — есть `method`, но нет `id` (ответа не ждёт);
- **ответ** — нет `method`, есть `id` и либо `result`, либо `error` (с полями `code` и `message`).

Важно: номера `id` назначает каждая сторона независимо, поэтому запрос клиента с `id = 1` и запрос
сервера с `id = 1` — разные запросы. Ответ идёт в **противоположном** направлении относительно запроса
и несёт тот же `id`.

Для каждого запроса (от любой стороны) найдите ответ и определите итог. Таймаут — 5000 мс:

- `timeout` — ответа нет совсем **или** он пришёл позже чем через 5000 мс после запроса (строго больше);
- иначе `error`, если ответ содержит поле `error`;
- иначе `ok`.

Ответы, для которых в логе нет запроса (с тем же `id` от противоположной стороны), игнорируйте.

## Подсказки

- Разберите каждую строку через `json.loads(raw)`. Различайте типы по наличию ключей `method` и `id`
  (`'id' in msg`), а не по значению — `id` бывает равен 0 или 1.
- Заведите словарь ожидающих запросов с ключом `(направление запроса, id)`. Ответ с направлением `S->C` ищите
  по ключу `('C->S', id)`, а ответ `C->S` — по ключу `('S->C', id)`.
- Задержка — `ts_ms ответа − ts_ms запроса`. Статус вычисляйте после обхода всего лога; запросы без ответа
  получают `timeout` и пустые `latency_ms` и `error_code`.

## Ожидаемый формат выходного файла

CSV с колонками `line_no, id, method, requester, status, latency_ms, error_code` — по одной строке на запрос
в порядке `line_no` запроса. `line_no` — номер строки **запроса**, `requester` — `client` или `server`,
`status` — `ok`, `error` или `timeout`, `latency_ms` — задержка ответа (пусто, если ответа нет),
`error_code` — `code` из ответа с ошибкой (иначе пусто). Числа — целые.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/mcp_protocol
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.